# Kickstarter Funding Prediction — All 5 Embeddings: Hyperparameter Tuning (Kaggle)
### 5 Embeddings x 3 Models x 3 Tuning Methods = up to 45 Tuned Runs

Combines Word2Vec, SBERT, DistilBERT, BERT, and TFIDF tuning into a single Kaggle notebook.
Tuning only — no baseline/default-hyperparameter runs.

**Speed optimizations applied vs. the original per-embedding notebooks:**
- `n_estimators` / `iterations` capped at **≤500** (low risk — boosting gains flatten out well
  before 1000+ trees at these learning rates)
- `RandomizedSearchCV`: iterations reduced **8 → 5**
- `BayesSearchCV`: iterations reduced **6 → 4**
- CV folds: **kept at 3** (unchanged, as requested)
- `HalvingGridSearchCV`: grid trimmed to the two lowest-impact dimensions per model
  (subsample/colsample/reg_alpha/reg_lambda for XGBoost, l2_leaf_reg for CatBoost,
  min_samples_split/min_samples_leaf for Random Forest cut to 2 values each) **and**
  `n_estimators`/`iterations` grid values also capped to 2 low values — combined, this cuts
  total grid combinations roughly 4-8x while keeping `learning_rate`/`max_depth`/`depth`
  (the highest-impact params) at full range
- `factor` increased **3 → 4** so weaker candidates are eliminated faster each halving round

**Input (Kaggle dataset paths — update `DATA_DIR` below):**
`ML_train.csv`, `ML_test.csv`, `word2vec_embeddings.csv`, `sbert_embeddings.csv`,
`distilbert_embeddings.csv`, `bert_embeddings.csv`, `tfidf_embeddings.csv`

**Output:** `kickstarter_all5_tuned_results.csv`, `kickstarter_all5_best_params.json`


## 0. Environment Check (Kaggle GPU)

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!pip install -q scikit-optimize


name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.159.04
Tesla T4, 15360 MiB, 580.159.04


## 1. Imports

In [2]:
import os
import json
import time
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    r2_score,
    make_scorer
)
from sklearn.experimental import enable_halving_search_cv  # noqa: F401 — required to unlock HalvingGridSearchCV
from sklearn.model_selection import RandomizedSearchCV, HalvingGridSearchCV, KFold

try:
    from xgboost import XGBRegressor
except ImportError:
    raise ImportError("xgboost is not installed. Run: pip install xgboost")

try:
    from catboost import CatBoostRegressor
except ImportError:
    raise ImportError("catboost is not installed. Run: pip install catboost")

try:
    from skopt import BayesSearchCV
    from skopt.space import Real, Integer, Categorical
    SKOPT_AVAILABLE = True
except ImportError:
    SKOPT_AVAILABLE = False
    print("scikit-optimize not installed — BayesSearchCV will be skipped.")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 200)

RANDOM_STATE = 42
USE_GPU = True  # set False to force CPU


## 2. Load Tabular Train/Test and All 5 Embedding Files

In [3]:
# Update this to your actual Kaggle input dataset directory.
DATA_DIR = "/kaggle/input/datasets/miftahullferdous/kickstarter-joint-embeddings/kickstarter_joint_embeddings"  # <-- update as needed

TRAIN_FILE = f"{DATA_DIR}/ML dataset/ML_train.csv"
TEST_FILE = f"{DATA_DIR}/ML dataset/ML_test.csv"

train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (16000, 81)
Test shape: (4000, 81)


In [4]:
EMBEDDING_FILES = {
    "Word2Vec":   f"{DATA_DIR}/NLP dataset/word2vec_embeddings.csv",
    "SBERT":      f"{DATA_DIR}/NLP dataset/sbert_embeddings.csv",
    "DistilBERT": f"{DATA_DIR}/NLP dataset/distilbert_embeddings.csv",
    "BERT":       f"{DATA_DIR}/NLP dataset/bert_embeddings.csv",
    "TFIDF":      f"{DATA_DIR}/NLP dataset/tfidf_embeddings.csv",
}

embeddings_raw = {}
for name, path in EMBEDDING_FILES.items():
    df = pd.read_csv(path)
    embeddings_raw[name] = df
    print(f"{name}: {df.shape}")


Word2Vec: (20000, 101)
SBERT: (20000, 385)
DistilBERT: (20000, 769)
BERT: (20000, 769)
TFIDF: (20000, 5001)


## 3. PCA Reduction for All 5 Embeddings (fit on train rows only)

Each embedding is reduced to 20 components to avoid dominating the ~80 existing tabular features.


In [5]:
N_COMPONENTS = 20

train_ids = set(train_df["id"])
embeddings_reduced = {}

for name, emb_df in embeddings_raw.items():
    feature_cols = [c for c in emb_df.columns if c != "id"]
    train_mask = emb_df["id"].isin(train_ids)

    pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
    pca.fit(emb_df.loc[train_mask, feature_cols])
    reduced_all = pca.transform(emb_df[feature_cols])

    reduced_cols = [f"{name.lower()}_pca_{i}" for i in range(N_COMPONENTS)]
    reduced_df = pd.DataFrame(reduced_all, columns=reduced_cols)
    reduced_df.insert(0, "id", emb_df["id"].values)
    embeddings_reduced[name] = reduced_df

    explained = pca.explained_variance_ratio_.sum()
    print(f"{name}: reduced to {N_COMPONENTS} dims, explained variance = {explained:.3f}")


Word2Vec: reduced to 20 dims, explained variance = 0.883
SBERT: reduced to 20 dims, explained variance = 0.684
DistilBERT: reduced to 20 dims, explained variance = 0.768
BERT: reduced to 20 dims, explained variance = 0.727
TFIDF: reduced to 20 dims, explained variance = 0.162


## 4. Evaluation Helper + Feature Builder

In [6]:
def evaluate_model(model_name, y_true_log, pred_log, actual_usd):
    pred_usd = np.expm1(pred_log)
    pred_usd = np.clip(pred_usd, a_min=0, a_max=None)

    mae_log = mean_absolute_error(y_true_log, pred_log)
    mse_log = mean_squared_error(y_true_log, pred_log)
    rmse_log = np.sqrt(mse_log)
    r2_log = r2_score(y_true_log, pred_log)

    mae_usd = mean_absolute_error(actual_usd, pred_usd)
    mse_usd = mean_squared_error(actual_usd, pred_usd)
    rmse_usd = np.sqrt(mse_usd)
    r2_usd = r2_score(actual_usd, pred_usd)

    rmsle = np.sqrt(mean_squared_log_error(actual_usd, pred_usd))

    return {
        "Model":      model_name,
        "MAE_log":    mae_log,
        "MSE_log":    mse_log,
        "RMSE_log":   rmse_log,
        "R2_log":     r2_log,
        "MAE_USD":    mae_usd,
        "MSE_USD":    mse_usd,
        "RMSE_USD":   rmse_usd,
        "R2_USD":     r2_usd,
        "RMSLE":      rmsle
    }, pred_usd


In [7]:
TARGET_RAW = "target_usd"
TARGET_LOG = "log_target"

DROP_FROM_X = [
    "id",
    "target_usd",
    "log_target",
    "goal_usd"
]

def build_features_for_embedding(embedding_name):
    reduced_df = embeddings_reduced[embedding_name]

    merged_train = train_df.merge(reduced_df, on="id", how="left")
    merged_test = test_df.merge(reduced_df, on="id", how="left")

    assert merged_train.shape[0] == train_df.shape[0], "Row count mismatch (train)"
    assert merged_test.shape[0] == test_df.shape[0], "Row count mismatch (test)"

    pca_cols = [c for c in reduced_df.columns if c != "id"]
    assert merged_train[pca_cols].isna().sum().sum() == 0
    assert merged_test[pca_cols].isna().sum().sum() == 0

    X_train = merged_train.drop(columns=DROP_FROM_X, errors="ignore")
    X_test = merged_test.drop(columns=DROP_FROM_X, errors="ignore")
    y_train = merged_train[TARGET_LOG].copy()
    y_test = merged_test[TARGET_LOG].copy()
    actual_usd = merged_test[TARGET_RAW].to_numpy()

    assert list(X_train.columns) == list(X_test.columns)
    return X_train, X_test, y_train, y_test, actual_usd


## 5. Hyperparameter Search Spaces (speed-optimized)

- `RandomizedSearchCV` and `BayesSearchCV` keep wide ranges (they sample sparsely, so width is cheap).
- `HalvingGridSearchCV` uses a **trimmed grid**: low-impact dimensions cut to 2 values,
  `n_estimators`/`iterations` capped to 2 low values, `learning_rate`/`max_depth`/`depth` kept
  at full range (highest-impact params).
- `n_estimators` / `iterations` capped at ≤500 everywhere (random/bayes ranges narrowed too).


In [8]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

CV = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)  # kept at 3 folds as requested
N_ITER_RANDOM = 5   # reduced from 8
N_ITER_BAYES = 4    # reduced from 6
HALVING_FACTOR = 4  # increased from 3 -> eliminates weak candidates faster


In [9]:
# ---- RandomizedSearchCV distributions (n_estimators/iterations capped at <=500) ----
xgb_param_dist = {
    "n_estimators":     [200, 300, 400, 500],
    "learning_rate":    [0.01, 0.02, 0.03, 0.05, 0.08],
    "max_depth":        [4, 5, 6, 7, 8, 9],
    "min_child_weight": [1, 2, 3, 5, 7],
    "subsample":        [0.7, 0.8, 0.85, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.85, 0.9, 1.0],
    "reg_alpha":        [0, 0.01, 0.05, 0.1, 0.5],
    "reg_lambda":       [0.5, 1.0, 1.5, 2.0, 3.0],
}

catboost_param_dist = {
    "iterations":    [200, 300, 400, 500],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
    "depth":         [4, 5, 6, 7, 8, 9],
    "l2_leaf_reg":   [1, 2, 3, 5, 7, 9],
}

rf_param_dist = {
    "n_estimators":      [150, 250, 350, 450],
    "max_depth":         [10, 15, 20, 25],
    "min_samples_split": [2, 4, 6, 8],
    "min_samples_leaf":  [1, 2, 3, 4],
    "max_features":      ["sqrt", "log2", 0.5, 0.7],
}

# ---- HalvingGridSearchCV grids: trimmed (#1 low-impact dims + #5 n_estimators cap, combined) ----
xgb_param_grid = {
    "n_estimators":     [300, 500],            # capped to 2 low values (#5)
    "learning_rate":    [0.01, 0.02, 0.03, 0.05, 0.08],  # kept full (high impact)
    "max_depth":        [4, 5, 6, 7, 8, 9],              # kept full (high impact)
    "subsample":        [0.8, 1.0],            # trimmed to 2 (#1 low-impact)
    "colsample_bytree": [0.8, 1.0],            # trimmed to 2 (#1 low-impact)
    "reg_alpha":        [0, 0.1],              # trimmed to 2 (#1 low-impact)
    "reg_lambda":       [1.0, 2.0],            # trimmed to 2 (#1 low-impact)
}

catboost_param_grid = {
    "iterations":    [300, 500],               # capped to 2 low values (#5)
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],  # kept full (high impact)
    "depth":         [4, 5, 6, 7, 8, 9],              # kept full (high impact)
    "l2_leaf_reg":   [1, 3, 5],                # trimmed to 3 (#1 low-impact)
}

rf_param_grid = {
    "n_estimators":      [250, 450],           # capped to 2 low values (#5)
    "max_depth":         [10, 15, 20, 25],     # kept full (high impact)
    "min_samples_split": [2, 4],               # trimmed to 2 (#1 low-impact)
    "min_samples_leaf":  [1, 2],               # trimmed to 2 (#1 low-impact)
    "max_features":      ["sqrt", "log2"],     # trimmed to 2 (#1 low-impact)
}

if SKOPT_AVAILABLE:
    xgb_bayes_space = {
        "n_estimators":     Integer(200, 500),
        "learning_rate":    Real(0.01, 0.1, prior="log-uniform"),
        "max_depth":        Integer(4, 9),
        "min_child_weight": Integer(1, 7),
        "subsample":        Real(0.7, 1.0),
        "colsample_bytree": Real(0.7, 1.0),
        "reg_alpha":        Real(1e-3, 0.5, prior="log-uniform"),
        "reg_lambda":       Real(0.5, 3.0),
    }
    catboost_bayes_space = {
        "iterations":    Integer(200, 500),
        "learning_rate": Real(0.01, 0.1, prior="log-uniform"),
        "depth":         Integer(4, 9),
        "l2_leaf_reg":   Real(1.0, 9.0),
    }
    rf_bayes_space = {
        "n_estimators":      Integer(150, 450),
        "max_depth":         Integer(5, 30),
        "min_samples_split": Integer(2, 8),
        "min_samples_leaf":  Integer(1, 4),
        "max_features":      Categorical(["sqrt", "log2"]),
    }


In [10]:
def make_base_estimator(model_name):
    if model_name == "XGBoost":
        return XGBRegressor(
            objective="reg:squarederror", eval_metric="rmse",
            tree_method="hist", device="cuda" if USE_GPU else "cpu",
            random_state=RANDOM_STATE,
        )
    elif model_name == "CatBoost":
        return CatBoostRegressor(
            loss_function="RMSE", eval_metric="RMSE",
            task_type="GPU" if USE_GPU else "CPU", devices="0" if USE_GPU else None,
            random_seed=RANDOM_STATE, verbose=False,
        )
    elif model_name == "Random Forest":
        return RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE)
    else:
        raise ValueError(model_name)


def make_search_estimator(model_name, method):
    base = make_base_estimator(model_name)
    search_n_jobs = 1  # avoid multiple processes contending for the same GPU

    if method == "random":
        param_dist = {"XGBoost": xgb_param_dist, "CatBoost": catboost_param_dist,
                       "Random Forest": rf_param_dist}[model_name]
        return RandomizedSearchCV(
            estimator=base, param_distributions=param_dist, n_iter=N_ITER_RANDOM,
            scoring=rmse_scorer, cv=CV, n_jobs=search_n_jobs,
            random_state=RANDOM_STATE, verbose=0, refit=True,
        )

    elif method == "grid":
        param_grid = {"XGBoost": xgb_param_grid, "CatBoost": catboost_param_grid,
                       "Random Forest": rf_param_grid}[model_name]
        return HalvingGridSearchCV(
            estimator=base, param_grid=param_grid,
            scoring=rmse_scorer, cv=CV, n_jobs=search_n_jobs,
            factor=HALVING_FACTOR, resource="n_samples", min_resources="exhaust",
            random_state=RANDOM_STATE, verbose=0, refit=True,
        )

    elif method == "bayes":
        if not SKOPT_AVAILABLE:
            return None
        search_space = {"XGBoost": xgb_bayes_space, "CatBoost": catboost_bayes_space,
                         "Random Forest": rf_bayes_space}[model_name]
        return BayesSearchCV(
            estimator=base, search_spaces=search_space, n_iter=N_ITER_BAYES,
            scoring=rmse_scorer, cv=CV, n_jobs=search_n_jobs,
            random_state=RANDOM_STATE, verbose=0, refit=True,
        )

    else:
        raise ValueError(method)


MODEL_NAMES = ["Random Forest", "XGBoost", "CatBoost"]
TUNING_METHODS = ["random", "grid", "bayes"] if SKOPT_AVAILABLE else ["random", "grid"]
TUNING_METHOD_LABELS = {
    "random": "RandomizedSearchCV",
    "grid":   "HalvingGridSearchCV",
    "bayes":  "BayesSearchCV",
}
print("Tuning methods to run:", [TUNING_METHOD_LABELS[m] for m in TUNING_METHODS])
print("USE_GPU:", USE_GPU)


Tuning methods to run: ['RandomizedSearchCV', 'HalvingGridSearchCV', 'BayesSearchCV']
USE_GPU: True


## 6. Run All Tuning Combinations (5 Embeddings x 3 Models x 3 Methods)

In [11]:
EMBEDDINGS_ORDER = list(embeddings_reduced.keys())


In [12]:
tuned_results = []
best_params_log = {}

for embedding_name in EMBEDDINGS_ORDER:
    print(f"\n{'='*70}")
    print(f"Embedding: {embedding_name}")
    print(f"{'='*70}")

    X_train, X_test, y_train, y_test, actual_usd = build_features_for_embedding(embedding_name)

    for model_name in MODEL_NAMES:
        for method in TUNING_METHODS:
            method_label = TUNING_METHOD_LABELS[method]
            print(f"  Tuning {model_name} via {method_label}...")
            start_time = time.time()

            search = make_search_estimator(model_name, method)
            if search is None:
                print(f"    Skipped ({method_label} unavailable)")
                continue

            search.fit(X_train, y_train)
            best_model = search.best_estimator_
            tuning_time = time.time() - start_time

            pred_log = best_model.predict(X_test)

            run_label = f"{embedding_name} + {model_name} ({method_label})"
            metrics, _ = evaluate_model(run_label, y_test, pred_log, actual_usd)
            metrics["Embedding"] = embedding_name
            metrics["Base_Model"] = model_name
            metrics["Tuning_Method"] = method_label
            metrics["Tuning_Time_Seconds"] = tuning_time
            metrics["Best_CV_RMSE_log"] = -search.best_score_

            tuned_results.append(metrics)
            best_params_log[f"{embedding_name}_{model_name}_{method}"] = dict(search.best_params_)

            print(f"    Best CV RMSE_log={-search.best_score_:.4f} | "
                  f"Test RMSLE={metrics['RMSLE']:.4f}, R2_log={metrics['R2_log']:.4f}, "
                  f"time={tuning_time:.1f}s")

print(f"\nTotal tuned runs completed: {len(tuned_results)}")



Embedding: Word2Vec
  Tuning Random Forest via RandomizedSearchCV...
    Best CV RMSE_log=2.0706 | Test RMSLE=2.0625, R2_log=0.5472, time=154.4s
  Tuning Random Forest via HalvingGridSearchCV...
    Best CV RMSE_log=2.1245 | Test RMSLE=2.1084, R2_log=0.5268, time=221.9s
  Tuning Random Forest via BayesSearchCV...
    Best CV RMSE_log=2.1374 | Test RMSLE=2.1215, R2_log=0.5209, time=51.1s
  Tuning XGBoost via RandomizedSearchCV...
    Best CV RMSE_log=1.8849 | Test RMSLE=1.8577, R2_log=0.6314, time=23.2s
  Tuning XGBoost via HalvingGridSearchCV...
    Best CV RMSE_log=1.9561 | Test RMSLE=1.9540, R2_log=0.5928, time=2066.3s
  Tuning XGBoost via BayesSearchCV...
    Best CV RMSE_log=1.8761 | Test RMSLE=1.8497, R2_log=0.6340, time=20.2s
  Tuning CatBoost via RandomizedSearchCV...
    Best CV RMSE_log=2.0674 | Test RMSLE=2.0685, R2_log=0.5439, time=37.9s
  Tuning CatBoost via HalvingGridSearchCV...
    Best CV RMSE_log=1.9263 | Test RMSLE=1.8972, R2_log=0.6151, time=2571.9s
  Tuning CatBoos

## 7. Combine, Display, and Save Results

In [13]:
tuned_df = pd.DataFrame(tuned_results)

tuned_df = tuned_df[
    [
        "Embedding", "Base_Model", "Tuning_Method", "Model",
        "MAE_log", "MSE_log", "RMSE_log", "R2_log",
        "MAE_USD", "MSE_USD", "RMSE_USD", "R2_USD",
        "RMSLE", "Best_CV_RMSE_log", "Tuning_Time_Seconds"
    ]
]
tuned_df = tuned_df.sort_values("RMSLE").reset_index(drop=True)
display(tuned_df)

print("\nBest result overall:")
display(tuned_df.iloc[[0]][["Embedding", "Base_Model", "Tuning_Method", "RMSLE", "R2_log"]])

print("\nBest result per embedding:")
display(tuned_df.sort_values("RMSLE").groupby("Embedding").first()[["Base_Model", "Tuning_Method", "RMSLE", "R2_log"]])

tuned_df.to_csv("kickstarter_all5_tuned_results.csv", index=False)
with open("kickstarter_all5_best_params.json", "w") as f:
    json.dump(best_params_log, f, indent=2, default=str)

print("\nSaved: kickstarter_all5_tuned_results.csv")
print("Saved: kickstarter_all5_best_params.json")


,Embedding,Base_Model,Tuning_Method,Model,MAE_log,MSE_log,RMSE_log,R2_log,MAE_USD,MSE_USD,RMSE_USD,R2_USD,RMSLE,Best_CV_RMSE_log,Tuning_Time_Seconds
0,BERT,XGBoost,BayesSearchCV,BERT + XGBoost (BayesSearchCV),1.116757,2.282908,1.510929,0.757002,14751.291051,1.005675e+10,100283.369418,0.147841,1.509727,1.537711,20.966647
1,BERT,XGBoost,RandomizedSearchCV,BERT + XGBoost (RandomizedSearchCV),1.128542,2.326011,1.525127,0.752414,14719.297536,9.778380e+09,98885.693162,0.171429,1.523647,1.556581,23.488733
2,BERT,CatBoost,BayesSearchCV,BERT + CatBoost (BayesSearchCV),1.152119,2.381566,1.543232,0.746500,14789.482393,1.028413e+10,101410.716512,0.128574,1.541757,1.568731,22.961436
3,BERT,CatBoost,HalvingGridSearchCV,BERT + CatBoost (HalvingGridSearchCV),1.191698,2.517762,1.586746,0.732003,15202.108865,1.071080e+10,103493.006328,0.092420,1.585178,1.597538,2880.149602
4,BERT,XGBoost,HalvingGridSearchCV,BERT + XGBoost (HalvingGridSearchCV),1.279048,2.923160,1.709725,0.688851,15602.394382,1.103272e+10,105036.744412,0.065142,1.708919,1.680044,2093.834925
5,DistilBERT,XGBoost,BayesSearchCV,DistilBERT + XGBoost (BayesSearchCV),1.284273,3.010437,1.735061,0.679561,15057.723916,1.022031e+10,101095.570786,0.133981,1.734867,1.788469,21.256761
6,DistilBERT,XGBoost,RandomizedSearchCV,DistilBERT + XGBoost (RandomizedSearchCV),1.296999,3.099175,1.760447,0.670116,15143.520718,1.045550e+10,102252.163591,0.114052,1.760150,1.803431,23.809997
7,DistilBERT,XGBoost,HalvingGridSearchCV,DistilBERT + XGBoost (HalvingGridSearchCV),1.321019,3.162858,1.778443,0.663337,15400.438056,1.061836e+10,103045.414768,0.100253,1.778425,1.808717,2040.439929
8,DistilBERT,CatBoost,BayesSearchCV,DistilBERT + CatBoost (BayesSearchCV),1.341277,3.243748,1.801041,0.654727,15365.838206,1.080183e+10,103931.854178,0.084707,1.800957,1.838894,23.120584
9,BERT,Random Forest,RandomizedSearchCV,BERT + Random Forest (RandomizedSearchCV),1.357460,3.394690,1.842469,0.638661,15586.469386,1.091569e+10,104478.197607,0.075058,1.842469,1.852164,147.709302



Best result overall:


,Embedding,Base_Model,Tuning_Method,RMSLE,R2_log
0,BERT,XGBoost,BayesSearchCV,1.509727,0.757002



Best result per embedding:


,Base_Model,Tuning_Method,RMSLE,R2_log
Embedding,,,,
BERT,XGBoost,BayesSearchCV,1.509727,0.757002
DistilBERT,XGBoost,BayesSearchCV,1.734867,0.679561
SBERT,XGBoost,BayesSearchCV,2.234654,0.468460
TFIDF,XGBoost,BayesSearchCV,2.137150,0.513711
Word2Vec,XGBoost,BayesSearchCV,1.849686,0.633984



Saved: kickstarter_all5_tuned_results.csv
Saved: kickstarter_all5_best_params.json


## Summary

Produced up to **45 tuned runs** (5 embeddings x 3 models x 3 tuning methods) in a single Kaggle
session, with speed optimizations applied to keep total runtime manageable within the 12-hour
GPU session limit:

- `n_estimators`/`iterations` capped at ≤500 across all search methods
- `RandomizedSearchCV`: 5 iterations (was 8)
- `BayesSearchCV`: 4 iterations (was 6)
- `HalvingGridSearchCV`: trimmed grid (low-impact dims cut to 2-3 values, `n_estimators`/`iterations`
  capped to 2 values), `factor=4` (was 3)
- CV folds: kept at 3 (unchanged)

Expected impact: minor degradation (roughly 0.01–0.05 RMSLE) vs. the original wider grids —
relative ranking across embeddings/models should hold.
